In [5]:
from google.cloud import bigquery

PROJECT = "bq-learning-510104"
client = bigquery.Client(project = PROJECT, location = "US")

def mb(n):
    return None if n is None else round(n /1024 /1024,2)

def check(sql):
    cfg = bigquery.QueryJobConfig(dry_run=True,use_query_cache=False)
    job = client.query(sql,job_config = cfg)
    print(f"sẽ quét: {mb(job.total_bytes_processed)} MB")

def stats(sql, max_mb=100):
    cfg = bigquery.QueryJobConfig(use_query_cache=False, maximum_bytes_billed = max_mb *1024 *1024)
    job = client.query(sql,job_config=cfg)
    df = job.result().to_dataframe()
    print(f"processed: {mb(job.total_bytes_processed)}MB | Billed: {mb(job.total_bytes_billed)} MB")
    return df


In [6]:
sql = """
SELECT
  ROUND(SUM(sale_price),2) AS net_revenue
  ,ROUND(SUM(gross_profit),2) AS gross_profit
  ,ROUND(SAFE_DIVIDE(SUM(gross_profit), SUM(sale_price)),4) AS margin
  ,COUNT(DISTINCT(order_id))
  ,ROUND(SAFE_DIVIDE(SUM(sale_price), COUNT(DISTINCT order_id)),2) AS avg_order_value
  ,MAX(order_date) AS data_through

FROM `bq-learning-510104.dbt_dev.fct_net_sales`
WHERE order_date >= '2026-01-01' and order_date <= '2026-12-31'
"""

check(sql)
df = stats(sql)
print(len(df))
df.T

sẽ quét: 1.38 MB
processed: 1.38MB | Billed: 10.0 MB
1


,0
net_revenue,982813.17
gross_profit,510954.59
margin,0.5199
f0_,11083
avg_order_value,88.68
data_through,2026-10-22


In [7]:
# Cell 3
# Mục đích: Liệt kê đối tượng trong dbt_dev kèm loại và số dòng để ghi vào README (không tạo query job, không tốn byte).
tables = list(client.list_tables(f"{PROJECT}.dbt_dev"))   # bản tóm tắt, chưa có số dòng
print(len(tables))                                        # kiểm tra số đối tượng trước khi lặp
for item in tables:
    t = client.get_table(item.reference)                  # thông tin đầy đủ của một bảng
    print(t.table_id, t.table_type, t.num_rows)

6
dim_date TABLE 3287
dim_products TABLE 29120
fct_net_sales TABLE 45285
fct_net_sales_inc TABLE 45107
stg_thelook__order_items VIEW 0
stg_thelook__products VIEW 0
